# 2.6 Generalized Target Prediction

This notebook demonstrates how to use `afids_pipeline.py` to train an AFID-to-target
coordinate predictor for any brain target, using the STN as a concrete example.

**Sections:**
1. Setup and paths
2. Training an `AfidsTargetPredictor`
3. Saving and loading a model
4. Running `cross_validate_pipeline()` for evaluation
5. Single-subject inference

## 1. Setup

In [ ]:
import sys
from pathlib import Path
from glob import glob
import numpy as np
import pandas as pd

# Add notebook directory to path so afidutils and afids_pipeline are importable
notebook_dir = Path().resolve()
sys.path.insert(0, str(notebook_dir))

import afidutils
from afids_pipeline import (
    AfidsTargetPredictor,
    train_from_fcsvs,
    predict_target_coordinates,
    cross_validate_pipeline,
)

# Adjust these paths to match your data layout
base_path = notebook_dir.parent / 'data' / 'coordinate'
stn_path  = base_path / 'STN'
afids_path = base_path / 'AFIDs'

output_model_path = str(notebook_dir / 'afids_stn_generalized.pkl')

print('Notebook dir:', notebook_dir)
print('Base data path:', base_path)

## 2. Load and pre-process data manually

This mirrors the logic used in notebook 2.4. You can skip this section and jump
to **Section 2b** if you prefer the `train_from_fcsvs()` convenience function.

In [ ]:
# Load STN coordinate files
stn_fcsv = sorted(glob(str(stn_path / '*/stn_mcp/*/*/*MCP*.fcsv')))
afids_fcsv = sorted(glob(str(afids_path / '*/afids_mcp/*/*/*MCP*.fcsv')))

print(f'STN files: {len(stn_fcsv)}')
print(f'AFID files: {len(afids_fcsv)}')

# --- STN targets ---
df_stn = pd.concat([afidutils.fcsvtodf(s)[0] for s in stn_fcsv], axis=0)
df_stn.columns = ['RSTNx', 'LSTNx', 'RSTNy', 'LSTNy', 'RSTNz', 'LSTNz']
df_stn['LSTNx'] *= -1
df_stn_r = df_stn[['RSTNx', 'RSTNy', 'RSTNz']].rename(columns=lambda c: c[1:])
df_stn_l = df_stn[['LSTNx', 'LSTNy', 'LSTNz']].rename(columns=lambda c: c[1:])
df_stn_proc = pd.concat([df_stn_r, df_stn_l], axis=0)

# --- AFIDs features ---
df_afids_raw = pd.concat([afidutils.fcsvtodf(f)[0] for f in afids_fcsv], axis=0)
df_afids_proc = afidutils.flip_and_concatenate_hemispheres(df_afids_raw)

# Align AFIDs to STN subjects
common_idx = df_stn_proc.index.intersection(df_afids_proc.index)
X = df_afids_proc.loc[common_idx].to_numpy()
y = df_stn_proc.loc[common_idx].to_numpy()
groups = afidutils.create_list(X.shape[0] // 2)

print(f'X shape: {X.shape}, y shape: {y.shape}')

## 3. Train an `AfidsTargetPredictor`

In [ ]:
predictor = AfidsTargetPredictor(target_name='STN', pca_variance=0.99)
predictor.fit(X, y, groups=groups)

print(f'PCA retains {predictor.pca.n_components_} components')
print(f'Ridge alpha (x): {predictor.ridge_x.alpha:.4f}')
print(f'Ridge alpha (y): {predictor.ridge_y.alpha:.4f}')
print(f'Ridge alpha (z): {predictor.ridge_z.alpha:.4f}')

## 4. Save and load the model

In [ ]:
# Save
predictor.save(output_model_path)

# Load back
loaded_predictor = AfidsTargetPredictor.load(output_model_path)
print(f'Loaded predictor for: {loaded_predictor.target_name}')

## 5. Cross-validate the pipeline

`cross_validate_pipeline()` runs a full repeated group K-fold evaluation and returns
a DataFrame with per-fold metrics.

In [ ]:
cv_results = cross_validate_pipeline(
    X_afids=X,
    y_target=y,
    groups=groups,
    n_splits=4,
    n_repeats=10,
    pca_variance=0.99,
)

print(cv_results.describe().round(3))
print(f"\nMean ED: {cv_results['ED'].mean():.3f} ± {cv_results['ED'].std():.3f} mm")

## 6. Single-subject inference

`predict_target_coordinates()` wraps the full pipeline for a single subject.

In [ ]:
# Replace with an actual subject AFID fcsv path
example_afid_fcsv = afids_fcsv[0]

y_pred = predict_target_coordinates(example_afid_fcsv, output_model_path)
print(f'Predicted coordinates (right, left) for {Path(example_afid_fcsv).name}:')
print(f'  Right: x={y_pred[0, 0]:.3f}  y={y_pred[0, 1]:.3f}  z={y_pred[0, 2]:.3f}')
print(f'  Left:  x={y_pred[1, 0]:.3f}  y={y_pred[1, 1]:.3f}  z={y_pred[1, 2]:.3f}')

## 7. Using `train_from_fcsvs()` convenience function

For a new target, you can use `train_from_fcsvs()` to go from raw fcsv files to
a saved model in one call. The `target_col_names` argument must list the six column
names in the order: right-x, left-x, right-y, left-y, right-z, left-z.

In [ ]:
# Example: train from file lists (using STN data as demonstration)
# Uncomment and adapt paths to use for a different target.

# trained_predictor = train_from_fcsvs(
#     afids_fcsv_list=afids_fcsv,
#     target_fcsv_list=stn_fcsv,
#     target_col_names=['RSTNx', 'LSTNx', 'RSTNy', 'LSTNy', 'RSTNz', 'LSTNz'],
#     output_model_path=output_model_path,
#     target_name='STN',
#     pca_variance=0.99,
# )
print('Uncomment the block above and set your file paths to use train_from_fcsvs().')